In [1]:
import pandas as pd

In [5]:
df = pd.read_csv('labels.csv')
for row in df.rolling(1):
    print(row['audio_path'][0])

300_AUDIO.wav


In [ ]:
str(df['participant_id'])}_P', str(path['audio_path'])

# Data cleaning

### first we clean audio files to remove background noises and enhance speaker voices

we can test to see if this makes things better

In [ ]:
import os
import subprocess

INPUT_FILE = r"/Users/gurusai/Desktop/DAIC_Raw/300_P/300_AUDIO.wav"
OUTPUT_DIR = "data"

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Step 1: Demucs
print("Running Demucs...")
subprocess.run([
    "demucs",
    "--two-stems=vocals",
    INPUT_FILE
])

base = os.path.splitext(os.path.basename(INPUT_FILE))[0]
demucs_out = f"separated/htdemucs/{base}/vocals.wav"

# Step 2: FFmpeg cleaning
output_path = os.path.join(OUTPUT_DIR, "cleanedv2.wav")

print("Applying FFmpeg filters...")

subprocess.run([
    "ffmpeg",
    "-i", demucs_out,
    "-af",
    # Chain of filters:
    # 1. highpass → remove low rumble
    # 2. lowpass → remove harsh high freq noise
    # 3. afftdn → noise reduction
    # 4. compand → smooth dynamics (makes speech clearer)
    "highpass=f=80, lowpass=f=12000, afftdn=nf=-22:tn=1, equalizer=f=3000:t=q:w=1:g=3, compand",
    output_path,
    "-y"
])

print(f"Done! Output saved at: {output_path}")

# second we get rid of the interviewer segments of the convo and encode them separately

In [ ]:
import pandas as pd
import os

import librosa
import soundfile as sf
import os
import numpy as np
def _normalize_visual_df(df, prefix):
    # Some OpenFace exports use leading spaces in headers.
    df = df.rename(columns={c: c.strip() for c in df.columns}).copy()

    required_keys = ["frame", "timestamp"]
    missing = [c for c in required_keys if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns {missing} in {prefix} file")

    # Drop quality-control columns from every visual dataset.
    drop_cols = [c for c in ["confidence", "success"] if c in df.columns]
    if drop_cols:
        df = df.drop(columns=drop_cols)

    # Prefix all remaining non-join columns to avoid collisions.
    feature_cols = [c for c in df.columns if c not in required_keys]
    rename_map = {c: f"{prefix}_{c}" for c in feature_cols}
    return df.rename(columns=rename_map)

def process_full_multimodal_alignment(p_id, p_folder, target_sr=16000,output_dir="data"):
    # 1. Load transcript
    df_t = pd.read_csv(os.path.join(p_folder, f"{p_id}_TRANSCRIPT.csv"), sep='\t')

    # 2. Load OpenFace streams
    df_clnf = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_features3D.txt"))
    df_gaze = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_gaze.txt"))
    df_pose = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_pose.txt"))
    df_au = pd.read_csv(os.path.join(p_folder, f"{p_id}_CLNF_AUs.txt"))

    # 3. Normalize and disambiguate columns before merging
    df_clnf = _normalize_visual_df(df_clnf, "clnf")
    df_gaze = _normalize_visual_df(df_gaze, "gaze")
    df_pose = _normalize_visual_df(df_pose, "pose")
    df_au = _normalize_visual_df(df_au, "au")

    # Merge visual features on frame+timestamp into one master table.
    df_visual = (
        df_clnf
        .merge(df_gaze, on=["frame", "timestamp"], how="inner")
        .merge(df_pose, on=["frame", "timestamp"], how="inner")
        .merge(df_au, on=["frame", "timestamp"], how="inner")
    )

    processed_turns = []
    last_question = "Initial Greeting"

    # audio processing params
    raw_audio_path = os.path.join(p_folder, f"{p_id}_AUDIO.wav")
    cumulative_time = 0.0
    manifest = []
    # Load the full audio once to save I/O overhead
    # We use sr=None to get original, then resample to 16k for the model
    y, sr = librosa.load(raw_audio_path, sr=target_sr)
    
    stitched_audio = []

    # Handle slight transcript schema variations.
    stop_col = "stop_time" if "stop_time" in df_t.columns else "end_time"

    for _, row in df_t.iterrows():
        text = str(row["value"]).lower()

        # Interviewer turn updates context anchor.
        if row["speaker"] == "Ellie":
            if "?" in text or any(q in text for q in ["how", "why", "tell me", "describe", "do you", "did you", "can you", "when", "have you"]):
                if len(text.split()) > 3:
                    last_question = text
            continue

        # Participant turn gets aligned to visual window.
        if row["speaker"] == "Participant":
            start, stop = row["start_time"], row[stop_col]
            
            visual_mask = (df_visual["timestamp"] >= start) & (df_visual["timestamp"] <= stop)
            turn_visual_data = df_visual[visual_mask]

            duration = stop - start
            new_start = cumulative_time
            new_stop = cumulative_time + duration
            start_sample = int(start * target_sr)
            stop_sample = int(stop * target_sr)
            segment = y[start_sample:stop_sample]
            stitched_audio.extend(segment)

            if not turn_visual_data.empty:
                processed_turns.append({
                    "participant_id": p_id,
                    "question_context": last_question,
                    "answer_text": text,
                    "start_time": start,
                    "stop_time": stop,
                    "visual_features": turn_visual_data.drop(columns=["frame", "timestamp"]).values,
                    'stitched_audio_start': new_start,
                    'stitched_audio_stop': new_stop,
                })
            cumulative_time += duration

    print(f"Visual columns: {len(df_visual.columns)}")
    return processed_turns

# Example usage:
p_id = "301"
p_folder = r"/Users/gurusai/Desktop/DAIC_Raw/301_P"
aligned_data = process_full_multimodal_alignment(p_id, p_folder)
print(f"Aligned participant turns: {len(aligned_data)}")

Visual columns: 244
Aligned participant turns: 104


In [17]:
import numpy as np

In [23]:
import librosa
import soundfile as sf
import os
import numpy as np

def snip_participant_audio(p_id, p_folder, participant_turns, output_dir, target_sr=16000):
    """
    p_id: Participant ID (e.g., '300')
    p_folder: Path to the specific participant's folder
    participant_turns: The list of dicts we created in the previous step
    """
    raw_audio_path = os.path.join(p_folder, f"{p_id}_AUDIO.wav")
    cumulative_time = 0.0
    manifest = []
    # Load the full audio once to save I/O overhead
    # We use sr=None to get original, then resample to 16k for the model
    y, sr = librosa.load(raw_audio_path, sr=target_sr)
    
    stitched_audio = []
    
    for turn in participant_turns:
        start_time = turn['start_time']
        stop_time = turn['stop_time']
        duration = turn['stop_time'] - turn['start_time']
        
        # Define the window in the NEW "Super-Cut" file
        new_start = cumulative_time
        new_stop = cumulative_time + duration
        
        # Convert seconds to sample indices
        start_sample = int(start_time * target_sr)
        stop_sample = int(stop_time * target_sr)
        
        # Extract the segment
        segment = y[start_sample:stop_sample]
        stitched_audio.extend(segment)
        manifest.append({
            'original_start': turn['start_time'],
            'new_start': new_start,
            'new_stop': new_stop,
            'text': turn['answer_text'],
            'context': turn['question_context']
        })
        cumulative_time += duration

        
    # Convert list back to numpy array
    stitched_audio = np.array(stitched_audio)
    
    # Save the 'Clean' Participant-Only Audio
    output_path = os.path.join(output_dir, f"{p_id}_CLEAN_AUDIO.wav")
    sf.write(output_path, stitched_audio, target_sr)
    
    print(f"✅ Saved stitched audio for {p_id}: {len(stitched_audio)/target_sr:.2f}s")
    return output_path, manifest

#usage:
output_dir = "data"
os.makedirs(output_dir, exist_ok=True)
clean_audio_path, manifest = snip_participant_audio(p_id, p_folder, aligned_data, output_dir)

✅ Saved stitched audio for 301: 475.44s


In [24]:
pd.DataFrame(manifest)

,original_start,new_start,new_stop,text,context
0,32.738,0.00,0.33,thank you,Initial Greeting
1,42.088,0.33,0.76,mmm k,Initial Greeting
2,54.328,0.76,2.19,i'm doing good thank you,how are you doing today
3,59.858,2.19,3.28,i'm from los angeles,how are you doing today
4,63.538,3.28,3.85,oh great,how are you doing today
...,...,...,...,...,...
99,784.788,468.38,473.61,i don't know i guess that's basically what i t...,how would your best friend describe you
100,797.918,473.61,474.03,okay,how would your best friend describe you
101,800.678,474.03,474.63,no problem,how would your best friend describe you
102,802.388,474.63,475.07,alright,how would your best friend describe you
